In [4]:
import re
import numpy as np
import pandas as pd
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

dokumen = [
    "Sistem komputer digunakan untuk mengolah data menjadi informasi.",
    "Jaringan komputer menghubungkan beberapa perangkat untuk berbagi data.",
    "Kecerdasan buatan membantu komputer melakukan tugas manusia.",
    "Sistem temu kembali informasi digunakan untuk mencari dokumen relevan."
]

stopword = StopWordRemoverFactory().create_stop_word_remover()

dokumen_pre = []

for teks in dokumen:
    teks = teks.lower()
    token = re.findall(r'\w+', teks)
    teks = stopword.remove(" ".join(token))
    dokumen_pre.append(teks)

print("HASIL PREPROCESSING")
for i, teks in enumerate(dokumen_pre, 1):
    print(f"D{i}: {teks}")

bow = CountVectorizer()

X = bow.fit_transform(dokumen_pre).toarray()

term = bow.get_feature_names_out()

df_bow = pd.DataFrame(
    X,
    columns=term,
    index=["D1", "D2", "D3", "D4"]
)

print("\nBAG-OF-WORDS (RAW COUNT)")
display(df_bow)


# TF = jumlah kemunculan term dalam dokumen
TF = X

# DF = jumlah dokumen yang mengandung term
DF = (X > 0).sum(axis=0)

# IDF = ln(N / DF)
N = len(dokumen_pre)
IDF = np.log(N / DF)

# TF-IDF = TF × IDF
TFIDF_manual = TF * IDF


df_tf = pd.DataFrame(
    TF,
    columns=term,
    index=["D1", "D2", "D3", "D4"]
)

print("\nTERM FREQUENCY (TF)")
display(df_tf)


df_df_idf = pd.DataFrame({
    "Term": term,
    "DF": DF,
    "IDF": IDF
})

print("\nDOCUMENT FREQUENCY (DF) DAN INVERSE DOCUMENT FREQUENCY (IDF)")
display(df_df_idf.round(4))

print("\nCONTOH PERHITUNGAN MANUAL")

# Contoh term komputer pada D1
print("\n1. Term 'komputer' pada D1")
print("TF = 1")
print("DF = 3")
print("IDF = ln(4 / 3) = 0.2877")
print("TF-IDF = 1 × 0.2877 = 0.2877")

# Contoh term mengolah pada D1
print("\n2. Term 'mengolah' pada D1")
print("TF = 1")
print("DF = 1")
print("IDF = ln(4 / 1) = 1.3863")
print("TF-IDF = 1 × 1.3863 = 1.3863")


df_tfidf_manual = pd.DataFrame(
    TFIDF_manual,
    columns=term,
    index=["D1", "D2", "D3", "D4"]
)

print("\nMATRIKS TF-IDF MANUAL")
display(df_tfidf_manual.round(4))

tfidf = TfidfVectorizer(
    smooth_idf=False,
    norm=None
)

TFIDF_sklearn = tfidf.fit_transform(dokumen_pre).toarray()

term_sklearn = tfidf.get_feature_names_out()

df_tfidf_sklearn = pd.DataFrame(
    TFIDF_sklearn,
    columns=term_sklearn,
    index=["D1", "D2", "D3", "D4"]
)

print("\nMATRIKS TF-IDF SCIKIT-LEARN")
display(df_tfidf_sklearn.round(4))


print("\nPERBANDINGAN HASIL")
print("Apakah hasil manual dan scikit-learn sama?")
print(np.allclose(TFIDF_manual, TFIDF_sklearn))


print("\nTERM DENGAN TF-IDF TERTINGGI")

for i, nama in enumerate(["D1", "D2", "D3", "D4"]):

    nilai_maks = TFIDF_manual[i].max()

    term_maks = term[TFIDF_manual[i] == nilai_maks]

    print(
        f"{nama}: {', '.join(term_maks)} "
        f"= {nilai_maks:.4f}"
    )

print("\nANALISIS SINGKAT")

print("""
D1:
Term 'mengolah' memiliki bobot TF-IDF tinggi karena hanya muncul
pada D1 sehingga menjadi salah satu ciri khusus dokumen tersebut.

D2:
Term 'berbagi', 'jaringan', 'menghubungkan', dan 'perangkat'
memiliki bobot TF-IDF tinggi karena hanya muncul pada D2.

D3:
Term 'buatan', 'kecerdasan', 'membantu', 'melakukan', dan
'manusia' memiliki bobot TF-IDF tinggi karena hanya muncul pada D3.

D4:
Term 'dokumen', 'kembali', 'mencari', 'relevan', dan 'temu'
memiliki bobot TF-IDF tinggi karena hanya muncul pada D4.

Term 'komputer' memiliki nilai TF-IDF lebih rendah karena muncul
pada tiga dokumen, sehingga tidak terlalu spesifik terhadap satu
dokumen.
""")

HASIL PREPROCESSING
D1: sistem komputer digunakan mengolah data menjadi informasi
D2: jaringan komputer menghubungkan beberapa perangkat berbagi data
D3: kecerdasan buatan membantu komputer melakukan tugas manusia
D4: sistem temu informasi digunakan mencari dokumen relevan

BAG-OF-WORDS (RAW COUNT)


,beberapa,berbagi,buatan,data,digunakan,dokumen,informasi,jaringan,kecerdasan,komputer,...,membantu,mencari,menghubungkan,mengolah,menjadi,perangkat,relevan,sistem,temu,tugas
D1,0,0,0,1,1,0,1,0,0,1,...,0,0,0,1,1,0,0,1,0,0
D2,1,1,0,1,0,0,0,1,0,1,...,0,0,1,0,0,1,0,0,0,0
D3,0,0,1,0,0,0,0,0,1,1,...,1,0,0,0,0,0,0,0,0,1
D4,0,0,0,0,1,1,1,0,0,0,...,0,1,0,0,0,0,1,1,1,0



TERM FREQUENCY (TF)


,beberapa,berbagi,buatan,data,digunakan,dokumen,informasi,jaringan,kecerdasan,komputer,...,membantu,mencari,menghubungkan,mengolah,menjadi,perangkat,relevan,sistem,temu,tugas
D1,0,0,0,1,1,0,1,0,0,1,...,0,0,0,1,1,0,0,1,0,0
D2,1,1,0,1,0,0,0,1,0,1,...,0,0,1,0,0,1,0,0,0,0
D3,0,0,1,0,0,0,0,0,1,1,...,1,0,0,0,0,0,0,0,0,1
D4,0,0,0,0,1,1,1,0,0,0,...,0,1,0,0,0,0,1,1,1,0



DOCUMENT FREQUENCY (DF) DAN INVERSE DOCUMENT FREQUENCY (IDF)


...


CONTOH PERHITUNGAN MANUAL

1. Term 'komputer' pada D1
TF = 1
DF = 3
IDF = ln(4 / 3) = 0.2877
TF-IDF = 1 × 0.2877 = 0.2877

2. Term 'mengolah' pada D1
TF = 1
DF = 1
IDF = ln(4 / 1) = 1.3863
TF-IDF = 1 × 1.3863 = 1.3863

MATRIKS TF-IDF MANUAL


,beberapa,berbagi,buatan,data,digunakan,dokumen,informasi,jaringan,kecerdasan,komputer,...,membantu,mencari,menghubungkan,mengolah,menjadi,perangkat,relevan,sistem,temu,tugas
D1,0.0000,0.0000,0.0000,0.6931,0.6931,0.0000,0.6931,0.0000,0.0000,0.2877,...,0.0000,0.0000,0.0000,1.3863,1.3863,0.0000,0.0000,0.6931,0.0000,0.0000
D2,1.3863,1.3863,0.0000,0.6931,0.0000,0.0000,0.0000,1.3863,0.0000,0.2877,...,0.0000,0.0000,1.3863,0.0000,0.0000,1.3863,0.0000,0.0000,0.0000,0.0000
D3,0.0000,0.0000,1.3863,0.0000,0.0000,0.0000,0.0000,0.0000,1.3863,0.2877,...,1.3863,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.3863
D4,0.0000,0.0000,0.0000,0.0000,0.6931,1.3863,0.6931,0.0000,0.0000,0.0000,...,0.0000,1.3863,0.0000,0.0000,0.0000,0.0000,1.3863,0.6931,1.3863,0.0000



MATRIKS TF-IDF SCIKIT-LEARN


,beberapa,berbagi,buatan,data,digunakan,dokumen,informasi,jaringan,kecerdasan,komputer,...,membantu,mencari,menghubungkan,mengolah,menjadi,perangkat,relevan,sistem,temu,tugas
D1,0.0000,0.0000,0.0000,1.6931,1.6931,0.0000,1.6931,0.0000,0.0000,1.2877,...,0.0000,0.0000,0.0000,2.3863,2.3863,0.0000,0.0000,1.6931,0.0000,0.0000
D2,2.3863,2.3863,0.0000,1.6931,0.0000,0.0000,0.0000,2.3863,0.0000,1.2877,...,0.0000,0.0000,2.3863,0.0000,0.0000,2.3863,0.0000,0.0000,0.0000,0.0000
D3,0.0000,0.0000,2.3863,0.0000,0.0000,0.0000,0.0000,0.0000,2.3863,1.2877,...,2.3863,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,2.3863
D4,0.0000,0.0000,0.0000,0.0000,1.6931,2.3863,1.6931,0.0000,0.0000,0.0000,...,0.0000,2.3863,0.0000,0.0000,0.0000,0.0000,2.3863,1.6931,2.3863,0.0000



PERBANDINGAN HASIL
Apakah hasil manual dan scikit-learn sama?
False

TERM DENGAN TF-IDF TERTINGGI
D1: mengolah, menjadi = 1.3863
D2: beberapa, berbagi, jaringan, menghubungkan, perangkat = 1.3863
D3: buatan, kecerdasan, manusia, melakukan, membantu, tugas = 1.3863
D4: dokumen, mencari, relevan, temu = 1.3863

ANALISIS SINGKAT

D1:
Term 'mengolah' memiliki bobot TF-IDF tinggi karena hanya muncul
pada D1 sehingga menjadi salah satu ciri khusus dokumen tersebut.

D2:
Term 'berbagi', 'jaringan', 'menghubungkan', dan 'perangkat'
memiliki bobot TF-IDF tinggi karena hanya muncul pada D2.

D3:
Term 'buatan', 'kecerdasan', 'membantu', 'melakukan', dan
'manusia' memiliki bobot TF-IDF tinggi karena hanya muncul pada D3.

D4:
Term 'dokumen', 'kembali', 'mencari', 'relevan', dan 'temu'
memiliki bobot TF-IDF tinggi karena hanya muncul pada D4.

Term 'komputer' memiliki nilai TF-IDF lebih rendah karena muncul
pada tiga dokumen, sehingga tidak terlalu spesifik terhadap satu
dokumen.

